In [2]:
import pandas as pd
import numpy as np
import re
import gc
import sys
import random

# ================= 配置区域 =================
FILES = {
    "TaskA": "D:/RiboMotif-SF/match_result/taskA_pdb_matches.tsv",
    "Labels": "D:/RiboMotif-SF/match_result/motif_have_function.tsv"
}

OUTPUT_FILE = "D:/RiboMotif-SF/module/RiboMotif_Training_Set_V3.tsv"
RANDOM_SEED = 2025 

# === 关键修改：正负样本比例配置 ===
# 你的要求：正样本数量多一点，1.5:1 或 2:1
# 含义：每 1 个负样本，对应 1.5 个正样本 (Positives = 1.5 * Negatives)
RATIO_POS_TO_NEG = 1.5 

# ================= 核心清洗函数 (严格按照你的要求修改) =================

def clean_structure_custom(dbn_str):
    """
    标准化二级结构 - 定制版逻辑：
    1. 去除首尾空格
    2. 保留 { } [ ]
    3. 将 < > 转换为 ( )
    4. 将 A B a b 转换为 .
    5. 其他非法字符转换为 .
    """
    if pd.isna(dbn_str): return "."
    
    # 1. 去除看不见的空白符
    s = str(dbn_str).strip()
    
    # 2. 符号转换逻辑
    # < > -> ( )
    s = s.replace('<', '(').replace('>', ')')
    
    # A B a b -> . (你的要求)
    s = re.sub(r'[ABab]', '.', s)
    
    # 3. 兜底清洗：除了 () [] {} . 以外的字符全部变成 .
    # 这样既保留了你想要的假结符号，又清洗了杂质
    s = re.sub(r'[^()\[\]\{\}\.]', '.', s)
    
    return s

def clean_sequence(seq_str):
    """标准化序列：大写，去空白，T转U"""
    if pd.isna(seq_str): return "N"
    s = str(seq_str).upper()
    s = "".join(s.split()) # 移除所有中间空格
    s = s.replace('T', 'U')
    return s

# ================= 主程序 =================
def main():
    print(f"=== RiboMotif-SF Dataset Builder (Custom Rules) ===")
    print(f"Target Ratio (Pos:Neg) = {RATIO_POS_TO_NEG} : 1")

    # --- Step 1: 加载正样本 ID ---
    print(">>> [Step 1] Loading Functional Label IDs...")
    try:
        df_label = pd.read_csv(FILES['Labels'], sep='\t', usecols=['motif_id'])
        positive_ids = set(df_label['motif_id'].unique())
        print(f"    Loaded {len(positive_ids):,} functional IDs.")
        del df_label; gc.collect()
    except Exception as e:
        print(f"Error: {e}"); return

    # --- Step 2: 扫描提取指纹 ---
    print("\n>>> [Step 2] Scanning & Cleaning Data...")
    pos_fingerprints = set()
    neg_fingerprints = set()
    chunk_size = 500000
    rows_processed = 0
    
    for chunk in pd.read_csv(FILES['TaskA'], sep='\t', chunksize=chunk_size, 
                             usecols=['motif_id', 'context_seq', 'context_dbn']):
        
        # 清洗
        chunk['clean_seq'] = chunk['context_seq'].apply(clean_sequence)
        chunk['clean_dbn'] = chunk['context_dbn'].apply(clean_structure_custom) # 使用新逻辑
        
        # 质量控制 (QC)
        valid_mask = (chunk['clean_seq'].str.len() == chunk['clean_dbn'].str.len()) & \
                     (~chunk['clean_seq'].str.contains('N'))
        chunk = chunk[valid_mask]
        
        # 判定
        chunk['is_positive'] = chunk['motif_id'].isin(positive_ids)
        
        # 提取指纹
        pos_batch = set(zip(chunk.loc[chunk['is_positive'], 'clean_seq'], 
                            chunk.loc[chunk['is_positive'], 'clean_dbn']))
        neg_batch = set(zip(chunk.loc[~chunk['is_positive'], 'clean_seq'], 
                            chunk.loc[~chunk['is_positive'], 'clean_dbn']))
        
        pos_fingerprints.update(pos_batch)
        neg_fingerprints.update(neg_batch)
        
        rows_processed += len(chunk)
        sys.stdout.write(f"\r    Processed {rows_processed:,} rows... | Pos: {len(pos_fingerprints):,} | Neg: {len(neg_fingerprints):,}")
        sys.stdout.flush()

    # --- Step 3: 冲突解决 (Trust Positive) ---
    print("\n\n>>> [Step 3] Conflict Resolution...")
    conflicts = pos_fingerprints.intersection(neg_fingerprints)
    if len(conflicts) > 0:
        print(f"    Removing {len(conflicts):,} conflicts from Negative set.")
        neg_fingerprints = neg_fingerprints - conflicts
    
    n_pos_avail = len(pos_fingerprints)
    n_neg_avail = len(neg_fingerprints)
    print(f"    Available Positives: {n_pos_avail:,}")
    print(f"    Available Negatives: {n_neg_avail:,}")

    # --- Step 4: 类别平衡 (定制比例) ---
    print(f"\n>>> [Step 4] Balancing Classes (Pos:Neg ~ {RATIO_POS_TO_NEG}:1)...")
    
    # 逻辑：尽可能满足 1.5:1，同时不超出实际拥有的数据量
    # 假设我们要取尽可能多的数据
    
    # 情况A：正样本极多，受限于负样本 -> Pos = Neg * 1.5
    target_neg_a = n_neg_avail
    target_pos_a = int(target_neg_a * RATIO_POS_TO_NEG)
    
    # 情况B：负样本极多，受限于正样本 -> Neg = Pos / 1.5
    target_pos_b = n_pos_avail
    target_neg_b = int(target_pos_b / RATIO_POS_TO_NEG)
    
    # 决策：取两种情况中，能塞进可用数据池的最大子集
    if target_pos_a <= n_pos_avail:
        # 负样本是瓶颈，用完所有负样本
        final_n_neg = target_neg_a
        final_n_pos = target_pos_a
    else:
        # 正样本是瓶颈，用完所有正样本
        final_n_pos = target_pos_b
        final_n_neg = target_neg_b
        
    print(f"    Sampling Plan:")
    print(f"      - Positive: {final_n_pos:,} (out of {n_pos_avail:,})")
    print(f"      - Negative: {final_n_neg:,} (out of {n_neg_avail:,})")
    print(f"      - Actual Ratio: {final_n_pos/final_n_neg:.2f} : 1")

    random.seed(RANDOM_SEED)
    final_pos_list = random.sample(list(pos_fingerprints), final_n_pos)
    final_neg_list = random.sample(list(neg_fingerprints), final_n_neg)

    # --- Step 5: 构建与保存 ---
    print("\n>>> [Step 5] Saving...")
    df_pos = pd.DataFrame(final_pos_list, columns=['sequence', 'structure'])
    df_pos['label'] = 1
    df_neg = pd.DataFrame(final_neg_list, columns=['sequence', 'structure'])
    df_neg['label'] = 0
    
    df_final = pd.concat([df_pos, df_neg], ignore_index=True)
    df_final = df_final.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    # 最终检查
    print("    Structure Character Check (Sample 1000):")
    sample_structs = "".join(df_final['structure'].head(1000).tolist())
    print(f"    Chars present: {set(sample_structs)}")
    
    df_final.to_csv(OUTPUT_FILE, sep='\t', index=False)
    print(f"[DONE] Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

=== RiboMotif-SF Dataset Builder (Custom Rules) ===
Target Ratio (Pos:Neg) = 1.5 : 1
>>> [Step 1] Loading Functional Label IDs...
    Loaded 9,156 functional IDs.

>>> [Step 2] Scanning & Cleaning Data...
    Processed 27,064,697 rows... | Pos: 2,845,154 | Neg: 1,035,965

>>> [Step 3] Conflict Resolution...
    Removing 459,292 conflicts from Negative set.
    Available Positives: 2,845,154
    Available Negatives: 576,673

>>> [Step 4] Balancing Classes (Pos:Neg ~ 1.5:1)...
    Sampling Plan:
      - Positive: 865,009 (out of 2,845,154)
      - Negative: 576,673 (out of 576,673)
      - Actual Ratio: 1.50 : 1

>>> [Step 5] Saving...
    Structure Character Check (Sample 1000):
    Chars present: {')', '(', ']', '[', '{', '}', '.'}
[DONE] Saved to D:/RiboMotif-SF/module/RiboMotif_Training_Set_V3.tsv
